# Mamba from Scratch

This notebook implements the **Mamba** architecture from scratch for character-level language modeling. Mamba is a state-space model with a **selective mechanism** that allows it to focus on relevant information while ignoring noise.

## What We'll Learn

- How **state-space models (SSMs)** work as an alternative to RNNs
- The **selective mechanism** that makes Mamba powerful
- How to implement the core Mamba block from scratch
- Training a Mamba model for next-token prediction

## Key Differences from RNNs

- **RNNs**: Fixed recurrence with `h_t = tanh(W_x * x_t + W_h * h_{t-1})`
- **Mamba**: Selective SSM with input-dependent parameters that control information flow

## Configuration

All hyperparameters in one place for easy experimentation.

In [1]:
CONFIG = {
    # Data
    "block_size": 64,  # Sequence length for training
    "batch_size": 64,  # Number of sequences per batch
    
    # Model
    "n_embed": 128,  # Embedding dimension
    "d_model": 256,  # Hidden dimension (internal state size)
    "d_state": 16,  # SSM state dimension (N in paper)
    "d_conv": 4,  # Convolution kernel size
    "expand": 2,  # Expansion factor for inner dimension
    
    # Training
    "n_epochs": 50,  # Maximum training epochs
    "learning_rate": 3e-3,  # Optimizer learning rate
    "max_patience": 3,  # Early stopping patience
}

## Setup Device and Seed

Initialize device (CUDA/MPS/CPU) and set random seed for reproducibility.

In [ ]:
from aiml_notebooks import get_device, set_seed

set_seed(42)
device = get_device()
print(f"\nUsing device: {device}")

## Load and Inspect Data

We'll use the Lord of the Rings text for character-level language modeling.

In [2]:
with open("data/lotr.txt", "r", encoding="utf-8") as f:
    TEXT = f.read()

TEXT[:1000]

"####-SPECIAL NOTE: \n\n \nIn this reprint several minor inaccuracies, most of them noted by readers, have \nbeen corrected. For example, the rune text now corresponds exactly with the runes \non  Thror's  Map.  More  important  is  the  matter  of  Chapter  Five.  There  the  true \nstory  of  the  ending  of  the  Riddle  Game,  as  it  was  eventually  revealed  (under \npressure)  by Bilbo  to Gandalf,  is  now  given  according  to  the Red Book,  in  place \nof  the  version  Bilbo  first  gave  to  his  friends,  and  actually  set  down  in  his  diary. \nThis  departure  from  truth  on  the  part  of  a  most  honest  hobbit  was  a  portent  of \ngreat  significance.  It  does  not,  however,  concern  the  present  story,  and  those who \nin this edition make their first acquaintance with hobbit-lore need not troupe about \nit.  Its  explanation  lies in the history of the Ring, as it was set out in the chronicles \nof the Red Book of Westmarch, and is now told in The Lord

## Build Character-Level Tokenizer

Create vocabulary and encode/decode functions for character-level tokens.

In [3]:
VOCAB = sorted(list(set(TEXT)))
CONFIG["vocab_size"] = len(VOCAB)

ctoi = {c: i for i, c in enumerate(VOCAB)}
itoc = {i: c for i, c in enumerate(VOCAB)}

encode = lambda s: [ctoi[c] for c in s]
decode = lambda tokens: "".join([itoc[i] for i in tokens])

# Test encoding/decoding
test_str = "hello world"
print(f"Original: {test_str}")
print(f"Encoded: {encode(test_str)}")
print(f"Decoded: {decode(encode(test_str))}")
print(f"\nVocab size: {len(VOCAB)}")

Original: hello world
Encoded: [65, 62, 69, 69, 72, 1, 80, 72, 75, 69, 61]
Decoded: hello world

Vocab size: 99


## Tokenize Full Text

Convert the entire text into token IDs.

In [4]:
TOKENS = encode(TEXT)
print(f"Total tokens: {len(TOKENS):,}")
print(f"First 20 tokens: {TOKENS[:20]}")

Total tokens: 3,262,172
First 20 tokens: [4, 4, 4, 4, 11, 48, 45, 34, 32, 38, 30, 41, 1, 43, 44, 49, 34, 24, 1, 0]


## Create Dataset

Split the token sequence into fixed-size chunks for next-token prediction.

In [5]:
import torch
from torch.utils.data import Dataset

class ChunkedDataset(Dataset):
    def __init__(self, tokens, block_size=CONFIG["block_size"]):
        tokens = torch.tensor(tokens)
        
        # Calculate number of complete chunks
        n_chunks = len(tokens) // (block_size + 1)
        tokens = tokens[:n_chunks * (block_size + 1)]
        
        # Reshape into chunks
        self.chunks = tokens.view(n_chunks, block_size + 1)
    
    def __getitem__(self, idx):
        chunk = self.chunks[idx]
        x = chunk[:-1]  # Input: first block_size tokens
        y = chunk[1:]   # Target: shifted by 1
        return x, y
    
    def __len__(self):
        return len(self.chunks)

full_dataset = ChunkedDataset(TOKENS)
print(f"Dataset size: {len(full_dataset):,} chunks")
x_sample, y_sample = full_dataset[0]
print(f"Sample shapes - X: {x_sample.shape}, Y: {y_sample.shape}")

Dataset size: 50,187 chunks
Sample shapes - X: torch.Size([64]), Y: torch.Size([64])


## Dataset Split Wrapper

Create a wrapper to handle train/val splits without duplicating data.

In [6]:
class DatasetSplit(Dataset):
    def __init__(self, dataset, indices):
        self.dataset = dataset
        self.indices = indices
    
    def __getitem__(self, idx):
        dataset_idx = self.indices[idx]
        return self.dataset[dataset_idx]
    
    def __len__(self):
        return len(self.indices)

## Split Data

Create 80/20 train/validation split.

In [7]:
import random

indices = list(range(len(full_dataset)))
random.shuffle(indices)

split_idx = int(len(indices) * 0.8)
train_indices = indices[:split_idx]
val_indices = indices[split_idx:]

train_dataset = DatasetSplit(full_dataset, train_indices)
val_dataset = DatasetSplit(full_dataset, val_indices)

print(f"Train size: {len(train_dataset):,}")
print(f"Val size: {len(val_dataset):,}")

Train size: 40,149
Val size: 10,038


## Create Train DataLoader

Batch and shuffle training data.

In [8]:
from torch.utils.data import DataLoader

train_dataloader = DataLoader(
    train_dataset,
    shuffle=True,
    batch_size=CONFIG["batch_size"]
)

# Inspect a batch
for x, y in train_dataloader:
    print(f"Batch X shape: {x.shape}")
    print(f"Batch Y shape: {y.shape}")
    break

Batch X shape: torch.Size([64, 64])
Batch Y shape: torch.Size([64, 64])


## Create Validation DataLoader

Batch validation data without shuffling.

In [9]:
val_dataloader = DataLoader(
    val_dataset,
    shuffle=False,
    batch_size=CONFIG["batch_size"]
)

# Inspect a batch
for x, y in val_dataloader:
    print(f"Batch X shape: {x.shape}")
    print(f"Batch Y shape: {y.shape}")
    break

Batch X shape: torch.Size([64, 64])
Batch Y shape: torch.Size([64, 64])


## Mamba Architecture Theory

### State-Space Models (SSMs)

Traditional SSMs model sequences with continuous states:

```
h'(t) = A * h(t) + B * x(t)  # State evolution
y(t) = C * h(t) + D * x(t)   # Output
```

After discretization with step size Δ:

```
h_t = A_bar * h_{t-1} + B_bar * x_t
y_t = C * h_t
```

### Selective Mechanism

Mamba makes **B, C, and Δ input-dependent** (selective):

- **Δ(x)**: Controls how much new information to incorporate
- **B(x)**: Controls which inputs to let through
- **C(x)**: Controls how to read the state

This allows the model to:
1. **Focus** on relevant tokens
2. **Ignore** irrelevant information
3. **Adapt** dynamically to input content

## Implement SSM Block

The core selective state-space model computation.

In [10]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SelectiveSSM(nn.Module):
    def __init__(self, d_model, d_state):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state
        
        # SSM parameters
        # A: (d_model, d_state) - state transition matrix
        self.A_log = nn.Parameter(torch.randn(d_model, d_state))
        # D: (d_model,) - skip connection
        self.D = nn.Parameter(torch.ones(d_model))
        
        # Projections for selective parameters (input-dependent)
        self.x_proj = nn.Linear(d_model, d_state * 2 + 1, bias=False)
        
        # dt_proj projects from rank to d_model
        self.dt_proj = nn.Linear(1, d_model, bias=True)
    
    def forward(self, x):
        # x: (batch, L, D)
        batch_size, L, D = x.shape
        
        # Get A matrix (ensure it's negative for stability)
        A = -torch.exp(self.A_log)  # (D, N)
        
        # Project input to get selective parameters
        x_proj = self.x_proj(x)  # (batch, L, 2*N + 1)
        
        # Split into delta, B, C
        delta_raw = x_proj[:, :, 0:1]  # (batch, L, 1)
        B_ssm = x_proj[:, :, 1:self.d_state+1]  # (batch, L, N)
        C_ssm = x_proj[:, :, self.d_state+1:]  # (batch, L, N)
        
        # Project delta to d_model dimension and ensure positive
        delta = F.softplus(self.dt_proj(delta_raw))  # (batch, L, D)
        
        # Discretize: A_bar = exp(delta * A)
        # Broadcasting: delta (batch,L,D,1) * A (D,N) -> (batch,L,D,N)
        delta_A = torch.exp(delta.unsqueeze(-1) * A.unsqueeze(0).unsqueeze(0))  # (batch, L, D, N)
        
        # B_bar = delta * B
        # x: (batch,L,D,1), B: (batch,L,1,N), delta: (batch,L,D,1)
        delta_B = delta.unsqueeze(-1) * B_ssm.unsqueeze(2)  # (batch, L, D, N)
        
        # Scan: compute h_t = A_bar * h_{t-1} + B_bar * x_t
        h = torch.zeros(batch_size, D, self.d_state, device=x.device)  # (batch, D, N)
        ys = []
        
        for t in range(L):
            # h = A_bar[t] * h + B_bar[t] * x[t]
            h = delta_A[:, t] * h + delta_B[:, t] * x[:, t].unsqueeze(-1)  # (batch, D, N)
            # y = C[t] * h
            y = torch.einsum('bdn,bn->bd', h, C_ssm[:, t])  # (batch, D)
            ys.append(y)
        
        y = torch.stack(ys, dim=1)  # (batch, L, D)
        
        # Add skip connection
        y = y + self.D * x
        
        return y

## Implement Mamba Block

The full Mamba block with projections, convolution, SSM, and output projection.

In [11]:
class MambaBlock(nn.Module):
    def __init__(self, d_model, d_state, d_conv, expand):
        super().__init__()
        self.d_model = d_model
        self.d_inner = d_model * expand
        
        # Input projection (expand)
        self.in_proj = nn.Linear(d_model, self.d_inner * 2, bias=False)
        
        # Convolution (local context)
        self.conv1d = nn.Conv1d(
            in_channels=self.d_inner,
            out_channels=self.d_inner,
            kernel_size=d_conv,
            padding=d_conv - 1,
            groups=self.d_inner  # Depthwise convolution
        )
        
        # SSM block
        self.ssm = SelectiveSSM(self.d_inner, d_state)
        
        # Output projection (compress)
        self.out_proj = nn.Linear(self.d_inner, d_model, bias=False)
    
    def forward(self, x):
        # x: (B, L, D)
        B, L, D = x.shape
        
        # Input projection and split
        x_proj = self.in_proj(x)  # (B, L, 2*D_inner)
        x_proj, res = x_proj.chunk(2, dim=-1)  # Each (B, L, D_inner)
        
        # Convolution (transpose for conv1d: B, C, L)
        x_conv = self.conv1d(x_proj.transpose(1, 2))[:, :, :L].transpose(1, 2)  # (B, L, D_inner)
        
        # Activation
        x_conv = F.silu(x_conv)
        
        # SSM
        y = self.ssm(x_conv)  # (B, L, D_inner)
        
        # Gating with residual
        y = y * F.silu(res)
        
        # Output projection
        output = self.out_proj(y)  # (B, L, D)
        
        return output

## Implement Full Mamba Model

The complete model with embeddings, Mamba block, and language modeling head.

In [12]:
class Mamba(nn.Module):
    def __init__(
        self,
        vocab_size=CONFIG["vocab_size"],
        n_embed=CONFIG["n_embed"],
        d_model=CONFIG["d_model"],
        d_state=CONFIG["d_state"],
        d_conv=CONFIG["d_conv"],
        expand=CONFIG["expand"]
    ):
        super().__init__()
        
        # Embedding
        self.embeddings = nn.Embedding(vocab_size, n_embed)
        
        # Project to d_model if needed
        self.embed_proj = nn.Linear(n_embed, d_model) if n_embed != d_model else nn.Identity()
        
        # Mamba block
        self.mamba = MambaBlock(d_model, d_state, d_conv, expand)
        
        # Layer norm
        self.norm = nn.LayerNorm(d_model)
        
        # Language modeling head
        self.lm_head = nn.Linear(d_model, vocab_size)
    
    def forward(self, x):
        # x: (B, T)
        x_emb = self.embeddings(x)  # (B, T, n_embed)
        x_emb = self.embed_proj(x_emb)  # (B, T, d_model)
        
        # Mamba block
        h = self.mamba(x_emb)  # (B, T, d_model)
        
        # Norm
        h = self.norm(h)  # (B, T, d_model)
        
        # LM head
        logits = self.lm_head(h)  # (B, T, vocab_size)
        
        return logits

## Test Model Forward Pass

Verify the model produces correct output shapes.

In [ ]:
model = Mamba().to(device)

# Get a batch and test forward pass
x, _ = next(iter(train_dataloader))
x = x.to(device)
logits = model(x)

print(f"Input shape: {x.shape}")
print(f"Output shape: {logits.shape}")
print(f"Expected: (batch={CONFIG['batch_size']}, seq_len={CONFIG['block_size']}, vocab={CONFIG['vocab_size']})")

## Define Evaluation Function

Compute validation loss without gradient tracking.

In [ ]:
from tqdm import tqdm

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    losses = []
    
    for x, y in tqdm(loader, disable=True):
        x, y = x.to(device), y.to(device)
        logits = model(x)
        # Reshape for cross entropy
        logits = logits.reshape(-1, logits.size(-1))  # (B*T, V)
        y = y.reshape(-1)  # (B*T)
        loss = F.cross_entropy(logits, y)
        losses.append(loss.item())
    
    return sum(losses) / len(losses)

# Test evaluation
initial_loss = evaluate(model, val_dataloader)
print(f"Initial validation loss: {initial_loss:.4f}")

## Training Loop

Train the Mamba model with early stopping.

In [ ]:
import torch.optim as optim

n_epochs = CONFIG["n_epochs"]
lr = CONFIG["learning_rate"]
optimizer = optim.Adam(model.parameters(), lr=lr)

train_losses = []
val_losses = []
patience = max_patience = CONFIG["max_patience"]

for epoch in range(n_epochs):
    model.train()
    losses = []
    
    pbar = tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{n_epochs}")
    for x, y in pbar:
        # Move to device
        x, y = x.to(device), y.to(device)
        
        # Forward pass
        logits = model(x)
        logits = logits.reshape(-1, logits.size(-1))  # (B*T, V)
        y = y.reshape(-1)  # (B*T)
        loss = F.cross_entropy(logits, y)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        losses.append(loss.item())
        pbar.set_postfix({"train_loss": f"{loss.item():.4f}"})
    
    # Compute epoch metrics
    train_loss = sum(losses) / len(losses)
    val_loss = evaluate(model, val_dataloader)
    
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    
    print(f"Epoch {epoch+1}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}, patience={patience}")
    
    # Early stopping
    if len(val_losses) > 1:
        val_loss_delta = val_losses[-2] - val_losses[-1]
        if val_loss_delta > 0.01:
            patience = max_patience
        else:
            patience -= 1
            if patience == 0:
                print(f"\nEarly stopping triggered at epoch {epoch+1}")
                break

## Plot Training Curves

Visualize training and validation loss over epochs.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.plot(train_losses, label='Train Loss', marker='o', linewidth=2)
plt.plot(val_losses, label='Val Loss', marker='s', linewidth=2)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.title('Mamba Training Progress', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nFinal train loss: {train_losses[-1]:.4f}")
print(f"Final val loss: {val_losses[-1]:.4f}")

## Define Text Generation Function

Generate text autoregressively from a prompt.

In [ ]:
def generate(model, prompt, max_len=100):
    model.eval()
    tokens = encode(prompt)
    max_gen = max_len - len(tokens)
    assert max_gen > 0, "Prompt is longer than max_len"
    
    print(prompt, end="")
    
    with torch.no_grad():
        for _ in range(max_gen):
            # Forward pass
            tokens_t = torch.tensor(tokens).unsqueeze(0).to(device)  # (1, T)
            logits = model(tokens_t)  # (1, T, V)
            logits = logits[0, -1, :]  # (V,) - last token
            
            # Sample from distribution
            probs = F.softmax(logits, dim=-1)
            token = torch.multinomial(probs, num_samples=1).item()
            
            # Append and print
            tokens.append(token)
            print(decode([token]), end="")
    
    print()  # Newline at end

## Generate Sample Text

Test the trained model's text generation capabilities.

In [ ]:
generate(model, "Frodo picked up", max_len=150)

## Generate More Samples

Try different prompts to see what the model learned.

In [ ]:
prompts = [
    "The ring ",
    "Gandalf said ",
    "In the land of "
]

for prompt in prompts:
    print(f"\n{'='*80}")
    generate(model, prompt, max_len=100)

## Key Takeaways

### What We Learned

1. **State-Space Models**: SSMs model sequences through continuous state evolution, providing an alternative to RNN recurrence

2. **Selective Mechanism**: Making B, C, and Δ input-dependent allows the model to:
   - Focus on relevant information
   - Ignore noise
   - Adapt dynamically to content

3. **Mamba Architecture**:
   - Input projection with gating
   - Depthwise convolution for local context
   - Selective SSM for global sequence modeling
   - Output projection

4. **Advantages over RNNs**:
   - More efficient for long sequences
   - Better at selective attention
   - More stable training (no vanishing gradients)

### Next Steps

- Stack multiple Mamba blocks for deeper models
- Add residual connections between blocks
- Try on different datasets and tasks
- Compare performance with RNNs and Transformers